# Writing Why: write a story, and see why each word was chosen

A small Born-rule language model writes, and the **Born evidence per word (BEW)** says exactly how much each earlier
word contributed to the word it chose: the shares add up, to about $10^{-11}$, to the probability above its uniform
share (less a tiny query-only term). From the paper *Writing Why: Born-Rule Language Models with Exact Word
Evidence*. Sampling runs on a CPU; the exact evidence of the 16-pass model takes about 8 s per word on a laptop GPU
and a few minutes on a CPU.

In [ ]:
#@title Setup (run this first)
# Get the code (in Colab: a fresh copy of the repository; when opened from a clone: that clone)
import os, sys, subprocess
if os.path.exists("writing_why.py"):
    ROOT = ".."
elif os.path.exists("demo/writing_why.py"):
    ROOT = "."
else:
    if not os.path.exists("writing-why/.git"):
        subprocess.run(["git", "clone", "-q", "https://github.com/stein-research/writing-why"], check=True)
    ROOT = "writing-why"
ROOT = os.path.abspath(ROOT)
sys.path.insert(0, os.path.join(ROOT, "demo"))
import writing_why as ww
print("device:", ww.DEV)

## The paper's story

Appendix B's 16-pass story: draw 10 of 10 from seed 0. The checksum of its token ids should read `9b8ef1d6e5ba`
(on another GPU or CPU a word may rarely differ; the checksum tells you).

In [ ]:
m = ww.load("words_born-therm16p2_s0")       # the 16-pass model, TinyStories, seed 0
story = ww.write(m, "Once upon a time", words=250, temp=0.7, top_p=0.9, seed=0, draws=10, only=10, live=True)
print("\nchecksum", story.checksum, "(paper: 9b8ef1d6e5ba)", "| ended" if story.ended else "| cut")

## Why "ball"?

The second "ball" of the story, explained over every earlier word. Red raised its probability, blue lowered it; hover
over a word for its share.

In [ ]:
r = ww.explain(m, story.ids, story.find("ball", 2))
ww.show(story, r)

## Your own

Change the prompt, the seed or the word, and run again.

In [ ]:
PROMPT = "Once upon a time, there was a little cat"  #@param {type:"string"}
SEED = 1  #@param {type:"integer"}
mine = ww.write(m, PROMPT, words=120, seed=SEED, live=True)
print("\nchecksum", mine.checksum)

In [ ]:
WORD, NTH = "the", 2   # explain the NTH occurrence of WORD (it must come after the prompt's first word)
ww.show(mine, ww.explain(m, mine.ids, mine.find(WORD, NTH)))

## WikiText-103

The same model family trained on Wikipedia articles (numbers are split into digits, as in the dataset).

In [ ]:
w = ww.load("wiki_born-therm16p2_s0")
passage = ww.write(w, "The river", words=90, seed=0, draws=5, only=4, live=True)
print("\nchecksum", passage.checksum, "(paper: 7a33f541962d)")
ww.show(passage, ww.explain(w, passage.ids, passage.find("suburbs", 2)))